## Добавляем универсальные категории к исходному датасету

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

Сначала обработаем исходный датасет, дополненный названиями статей из справочника статей по id:

In [2]:
data_download = pd.read_csv("../data_download_new.csv")
articles_uc = pd.read_csv("articles_uc.csv")
articles = pd.read_csv("../articles.csv")

In [3]:
data_download = data_download.drop_duplicates('id', keep='first')
data_download = data_download[data_download['article_id'] != -1]

In [4]:
articles_inc = articles[(articles["type"] == "incoming") & (articles['id'].isin(data_download["article_id"].unique()))]

articles_inc['name'].nunique()

649

In [5]:

df = pd.merge(
    data_download,
    articles_inc[['id', 'name']],
    how='left',
    left_on='article_id',
    right_on='id'
)

df.drop(columns=['id_y'], inplace=True)
df.rename(columns={'id_x': 'id', 'name': 'article_name'}, inplace=True)

df['article_name'] = df['article_name'].str.lower()
df['universal_category'] = df['article_name'].map(articles_uc.set_index('raw')['universal_category'])

df['universal_category'].value_counts()


universal_category
пожертвования от физических лиц (напрямую)     323657
пожертвования через платформы                  179196
пожертвования от юридических лиц (напрямую)     11192
прочие недоходные операции                      10331
продажа услуг                                    8564
продажа товаров                                  6010
финансовые доходы                                2187
членские и учредительские взносы                  890
гранты субсидии конкурсы                          450
Name: count, dtype: int64

In [6]:
df.loc[df['article_id'] == 36389, 'universal_category'] = 'пожертвования от физических лиц (напрямую)'
df.loc[df['article_id'] == 47042, 'universal_category'] = 'пожертвования от физических лиц (напрямую)'
df.loc[df['id'] == 1038304, 'universal_category'] = 'пожертвования от физических лиц (напрямую)'
df.loc[df['id'] == 1046860, 'universal_category'] = 'пожертвования от юридических лиц (напрямую)'

df = df[df['article_id'] != 38767] # поступления по 1 рублю без опознавательных данных, может технические (проверка карты)
df.loc[df['article_id'] == 31833, 'universal_category'] = 'прочие недоходные операции'
df.loc[df['article_id'] == 46965, 'universal_category'] = 'прочие недоходные операции'
df.loc[df['article_id'] == 46967, 'universal_category'] = 'прочие недоходные операции'

df.loc[df['article_id'] == 47120, 'universal_category'] = 'пожертвования через платформы'
df.loc[df['article_id'] == 46966, 'universal_category'] = 'пожертвования через платформы'
df.loc[df['article_id'] == 45077, 'universal_category'] = 'пожертвования через платформы'
df.loc[df['article_id'] == 47041, 'universal_category'] = 'продажа услуг'
df.loc[df['article_id'] == 46969, 'universal_category'] = 'продажа услуг'
df.loc[df['article_id'] == 46888, 'universal_category'] = 'продажа товаров'
df.loc[df['article_id'] == 47043, 'universal_category'] = 'продажа товаров'

df.head()

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,projects__id,projects__user_id,projects__parent_id,counterparties__id,counterparties__user_id,counterparties__parent_id,robots__id,robots__user_id,article_alternative_names__user_id,article_name,universal_category
0,289,1,70,2022-09-08,1.0,NaN,0,incoming,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,290,1,71,2022-09-08,1.0,NaN,0,incoming,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,291,1,30,2022-09-08,1.0,NaN,0,incoming,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,295,1,72,2022-09-08,1.0,NaN,0,incoming,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,304,5,29,2022-09-19,1.0,NaN,301,incoming,13,0,0,0,0,5.0,39.0,301.0,39.0,299.0,13.0,39.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,грант,гранты субсидии конкурсы


In [7]:
# проверка заполнения категорий
display(df['id'].count()) # количество платежей в таблице
display(df['universal_category'].isna().sum()) # количество пропусков в категориях
df[df['article_id'] == 0]['id'].count() # количество нулевых(незаполеннных) значений статей - должно совпадать со значением выше

608415

65666

65666

In [8]:
df['universal_category'].value_counts()

universal_category
пожертвования от физических лиц (напрямую)     323802
пожертвования через платформы                  179310
пожертвования от юридических лиц (напрямую)     11193
прочие недоходные операции                      10335
продажа услуг                                    8568
продажа товаров                                  6014
финансовые доходы                                2187
членские и учредительские взносы                  890
гранты субсидии конкурсы                          450
Name: count, dtype: int64

In [9]:
df.to_parquet("data_download_uc.parquet", index=False)

Далее проверим датасет с исходно загруженными названиями статей, категорий доноров и проектов и раздадим универсальные категории аналогично схеме выше, данный датасет выгружался через месяц после предыдущего. Тут нет смысла отбирать статьи по справочнику и сверять по id с базой платежей, сразу ориентируемся на данные из базы платежей и названий статей в ней. Список универсальных категорий articles_uc_added.csv обновлен на базе предыдущей версии articles_uc.csv с учетом новых названий статей из обновленнного датасета платежей.

In [10]:
data = pd.read_csv("../data_download_09102025.csv")
articles_uc_added = pd.read_csv("articles_uc_added.csv")

In [11]:
print(data.id.count())

data = data.drop_duplicates('id', keep='first')
data = data[data['article_id'] != -1]

data.id.count()

646714


641679

In [12]:
data.articles__name.nunique()

748

In [13]:
data[~data.articles__name.str.lower().isin(articles_uc_added.raw.str.lower())].articles__name.unique()

array([nan], dtype=object)

In [14]:
data['universal_category'] = data['articles__name'].str.lower().map(articles_uc_added.set_index('raw')['universal_category'])

In [15]:
data.loc[data['article_id'] == 36389, 'universal_category'] = 'пожертвования от физических лиц (напрямую)'

In [16]:
# проверка заполнения категорий
display(data['id'].count()) # количество платежей в таблице
display(data['universal_category'].isna().sum()) # количество пропусков в категориях
data[data['article_id'] == 0]['id'].count() # количество нулевых(незаполеннных) значений статей - должно совпадать со значением выше

641679

64884

64884

In [17]:
data['universal_category'].value_counts()

universal_category
пожертвования от физических лиц (напрямую)     338125
пожертвования через платформы                  190881
пожертвования от юридических лиц (напрямую)     17061
прочие недоходные операции                      11007
продажа услуг                                    9328
продажа товаров                                  6609
финансовые доходы                                2351
членские и учредительские взносы                  931
гранты субсидии конкурсы                          502
Name: count, dtype: int64

In [18]:
data.to_parquet("data_download_09102025_uc.parquet", index=False)